# Extracción de embeddings con x-vector

En esta fase se extraerán embeddings de voz utilizando un modelo preentrenado tipo **x-vector**.

Los embeddings serán obtenidos para dos subconjuntos principales:

1. **Enrollment:** audios de referencia de cada locutor.
2. **Test:** audios de prueba que serán comparados contra los audios de enrollment.

Además, se utilizará una tabla de **trials**, la cual define los pares de comparación entre audios de enrollment y audios de test.

El objetivo final de esta fase es construir representaciones vectoriales de los audios que posteriormente serán utilizadas para calcular medidas de similitud entre locutores.

### Notación general

Sea $x_i$ una señal de audio correspondiente al audio \(i\).  
Un modelo x-vector puede entenderse como una función neuronal:

$$
f_{\theta}: x_i \longrightarrow z_i
$$

donde:

- $x_i$ es la señal de audio de entrada.
- $ f_{\theta} $ es el modelo x-vector preentrenado.
- $\theta$ representa los parámetros aprendidos del modelo.
- $z_i$ es el embedding extraído del audio.
- $z_i \in \mathbb{R}^{d}$, donde $d$ es la dimensión del embedding.

En términos prácticos, el embedding $z_i$ es un vector numérico que resume información relevante sobre la identidad vocal del locutor.

## 01. Librerías principales de esta fase

En esta etapa utilizaremos librerías para manipulación de datos, manejo de rutas, procesamiento de audio y extracción de embeddings mediante modelos neuronales.

### `torch`

torch es la librería principal de PyTorch.

PyTorch es un framework de deep learning que permite trabajar con tensores, redes neuronales y modelos entrenados.

En esta fase, torch es importante porque SpeechBrain está construido sobre PyTorch.

Un tensor es una estructura numérica similar a un arreglo de NumPy, pero con soporte para operaciones de deep learning y uso de GPU.

### `torchaudio`

torchaudio es una extensión de PyTorch especializada en procesamiento de audio.

Permite cargar audios directamente como tensores de PyTorch.

### `EncoderClassifier`

EncoderClassifier permite cargar modelos preentrenados para tareas de reconocimiento de locutor.

En esta fase se utilizará para cargar un modelo x-vector preentrenado.

### `Path`
Path permite trabajar con rutas de archivos de forma más robusta y legible.
Esto será importante antes de extraer embeddings, porque necesitamos confirmar que las rutas de audio sean accesibles desde el notebook.

#### `tqdm` 
tqdm permite mostrar barras de progreso en ciclos largos. Esto será útil cuando extraigamos embeddings de miles de audios.


In [1]:
# !pip install speechbrain

In [25]:
# Importamos las librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import torch
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier
from pathlib import Path
from tqdm import tqdm

c:\Users\Diego Max.000\anaconda3\Lib\site-packages\speechbrain\utils\torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [14]:
# Carga de datasets desde Hugging Face
from datasets import load_dataset
from datasets import Audio
import soundfile as sf

# Reproducción de audio dentro de Jupyter Notebook
import IPython.display as ipd

In [2]:
# Ruta del archivo con los audios de enrollment.
enrollment_csv_path = "enrollment.csv"

# Ruta del archivo con los audios de test.
test_csv_path = "test.csv"

# Ruta del archivo con los trials.
trials_csv_path = "trials.csv"

In [3]:
df_enrollment = pd.read_csv(enrollment_csv_path)
df_test = pd.read_csv(test_csv_path)
df_trials = pd.read_csv(trials_csv_path)

## 02. Carga del modelo x-vector de SpeechBrain

En esta etapa se cargará un modelo preentrenado tipo **x-vector** usando la librería SpeechBrain.

El modelo seleccionado es:

```python
speechbrain/spkrec-xvect-voxceleb

Este modelo fue entrenado para generar embeddings de locutor a partir de señales de voz.

In [4]:
# torch.cuda.is_available()
# Devuelve True si PyTorch detecta una GPU compatible con CUDA.
# Devuelve False si solo se usará CPU.

cuda_available = torch.cuda.is_available()

print("¿CUDA disponible?:", cuda_available)

if cuda_available:
    print("GPU detectada:", torch.cuda.get_device_name(0))
else:
    print("No se detectó GPU compatible con CUDA. Se usará CPU.")

¿CUDA disponible?: False
No se detectó GPU compatible con CUDA. Se usará CPU.


In [5]:
# En PyTorch, el dispositivo indica dónde se ejecutarán los tensores
# y el modelo: "cuda" para GPU o "cpu" para procesador.

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo seleccionado:", device)

Dispositivo seleccionado: cpu


### Nota sobre CPU y GPU

El modelo x-vector puede ejecutarse tanto en CPU como en GPU.

- Si se usa **CPU**, la extracción de embeddings funcionará, pero puede ser más lenta.
- Si se usa **GPU**, el procesamiento suele ser más rápido, especialmente para muchos audios.

Como en este proyecto se extraerán embeddings para más de 13 mil audios de test, utilizar GPU puede reducir considerablemente el tiempo de ejecución.

Sin embargo, el procedimiento experimental es el mismo en ambos casos.

In [7]:
# Creación de carpeta para modelos preentrenados

# Esta carpeta almacenará localmente los archivos descargados
# del modelo x-vector de SpeechBrain.

pretrained_model_dir = Path("pretrained_models")
pretrained_model_dir.mkdir(parents=True, exist_ok=True)

xvector_model_dir = pretrained_model_dir / "spkrec-xvect-voxceleb"

print("Carpeta del modelo:", xvector_model_dir)

Carpeta del modelo: pretrained_models\spkrec-xvect-voxceleb


In [ ]:
# !pip install "huggingface_hub==0.23.4"

  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.14.0
    Uninstalling huggingface_hub-1.14.0:
      Successfully uninstalled huggingface_hub-1.14.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 0.23.4 which is incompatible.


In [8]:
# Carga del modelo x-vector preentrenado


# EncoderClassifier.from_hparams(...)
#
# source:
#   Nombre del modelo disponible en Hugging Face/SpeechBrain.
#
# savedir:
#   Carpeta local donde se guardarán los archivos del modelo.
#
# run_opts:
#   Diccionario con opciones de ejecución.
#   En este caso se especifica el dispositivo: CPU o GPU.

classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb",
    savedir=str(xvector_model_dir),
    run_opts={"device": device}
)

print("Modelo x-vector cargado correctamente.")

c:\Users\Diego Max.000\anaconda3\Lib\site-packages\speechbrain\utils\torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
c:\Users\Diego Max.000\anaconda3\Lib\site-packages\speechbrain\utils\autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
c:\Users\Diego Max.000\anaconda3\Lib\inspect.py:1007: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speec

embedding_model.ckpt:   0%|          | 0.00/16.9M [00:00<?, ?B/s]

c:\Users\Diego Max.000\anaconda3\Lib\site-packages\speechbrain\utils\fetching.py:151: UserWarning: Using SYMLINK strategy on Windows for fetching potentially requires elevated privileges and is not recommended. See `LocalStrategy` documentation.
  warnings.warn(


mean_var_norm_emb.ckpt:   0%|          | 0.00/3.20k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/15.9M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

Modelo x-vector cargado correctamente.


In [9]:
# Revisión básica del objeto clasifier

print("Tipo de objeto cargado:")
print(type(classifier))

print("\nDispositivo utilizado por el modelo:")
print(device)

Tipo de objeto cargado:
<class 'speechbrain.inference.classifiers.EncoderClassifier'>

Dispositivo utilizado por el modelo:
cpu


## 03. Prueba de extracción de un solo embedding

En esta sección se hará la experimentación para la extracción de un solo embedding de un archivo de audio para familiarizarse con el proceso y posteriormente poder generalizarlo a todos los audios en la base de datos CIEMPIESS

In [10]:
# Selección de un audio de prueba desde enrollment

# Tomamos el primer audio de df_enrollment para verificar que:
# 1. La ruta existe.
# 2. El audio puede cargarse.
# 3. El modelo puede producir un embedding.

sample_row = df_enrollment.iloc[0]

sample_audio_id = sample_row["audio_id"]
sample_speaker_id = sample_row["speaker_id"]
sample_audio_path = sample_row["audio_path"]

print("Audio seleccionado:")
print("audio_id:", sample_audio_id)
print("speaker_id:", sample_speaker_id)
print("audio_path:", sample_audio_path)

Audio seleccionado:
audio_id: CMPL_F_01_12MAB_00651
speaker_id: F_01
audio_path: CMPL_F_01_12MAB_00651.flac


### Corrección metodológica: acceso a audios desde Hugging Face Datasets

En una etapa anterior se intentó cargar los audios directamente desde la columna `audio_path` mediante:

```python
torchaudio.load(audio_path)
```

Sin embargo, en este proyecto no se cuenta con una carpeta local descargada manualmente que contenga todos los archivos .flac de CIEMPIESS Light.Los audios fueron accedidos previamente mediante la librería datasets de Hugging Face.
Por esta razón, la extracción de embeddings x-vector no se realizará directamente desde rutas locales, sino desde los registros del objeto Dataset.


In [18]:
# Carga de CIEMPIESS Light desde Hugging Face

ciempiess = load_dataset(
    "ciempiess/ciempiess_light",
    split="train"
)

In [19]:
# Configurar columna de audio sin decodificación automática

ciempiess = ciempiess.cast_column(
    "audio",
    Audio(decode=False)
)

print("Columna de audio configurada con decode=False")

Columna de audio configurada con decode=False


In [ ]:
# Crear índice rápido audio_id -> posición en el Dataset

audio_id_to_index = {
    item["audio_id"]: idx
    for idx, item in enumerate(ciempiess)
}

print("Número de audio_id indexados:", len(audio_id_to_index))


Número de audio_id indexados: 16663


[('CMPL_F_32_11ANG_00003', 0),
 ('CMPL_F_32_11ANG_00002', 1),
 ('CMPL_F_32_11ANG_00019', 2),
 ('CMPL_F_32_11ANG_00015', 3),
 ('CMPL_F_32_11ANG_00010', 4)]

In [21]:
# Validar correspondencia entre metadatos y dataset Hugging Face

hf_audio_ids = set(audio_id_to_index.keys())

enrollment_ids = set(df_enrollment["audio_id"])
test_ids = set(df_test["audio_id"])

missing_enrollment_hf = enrollment_ids - hf_audio_ids
missing_test_hf = test_ids - hf_audio_ids

print("Audios de enrollment no encontrados en Hugging Face:", len(missing_enrollment_hf))
print("Audios de test no encontrados en Hugging Face:", len(missing_test_hf))

if len(missing_enrollment_hf) > 0:
    print("Ejemplos enrollment faltantes:")
    print(list(missing_enrollment_hf)[:10])

if len(missing_test_hf) > 0:
    print("Ejemplos test faltantes:")
    print(list(missing_test_hf)[:10])

Audios de enrollment no encontrados en Hugging Face: 0
Audios de test no encontrados en Hugging Face: 0


### Función `read_audio_from_hf_item`

La función `read_audio_from_hf_item()` recibe un registro individual del dataset de Hugging Face y devuelve la señal de audio como arreglo de NumPy.

Esta función contempla dos posibles formas de acceso al audio:

1. **Lectura desde bytes:** cuando el archivo de audio está disponible directamente en memoria.
2. **Lectura desde path:** cuando el dataset proporciona una ruta interna al archivo.

La función utilizada para leer el audio es:

```python
sf.read(...)
```
proveniente de la librería soundfile.

In [30]:

# Función para leer audio desde un item de Hugging Face


def read_audio_from_hf_item(item):
    """
    Lee la señal de audio desde un registro del dataset CIEMPIESS Light.

    La columna `audio` puede contener:
    1. Bytes del archivo de audio.
    2. Una ruta interna/cache al archivo de audio.

    Parámetros
    ----------
    item : dict
        Registro individual del dataset de Hugging Face.

    Retorna
    -------
    signal_np : np.ndarray
        Señal de audio como arreglo de NumPy.

    sampling_rate : int
        Frecuencia de muestreo del audio.
    """

    # Extraemos el diccionario asociado a la columna de audio.
    audio_info = item["audio"]

    # Obtenemos los bytes del audio, si están disponibles.
    audio_bytes = audio_info["bytes"]

    # Obtenemos la ruta interna/cache del audio, si está disponible.
    audio_path = audio_info["path"]

    # Caso 1: el audio está disponible como bytes.
    if audio_bytes is not None:
        signal_np, sampling_rate = sf.read(io.BytesIO(audio_bytes))

    # Caso 2: el audio se debe leer desde una ruta.
    else:
        signal_np, sampling_rate = sf.read(audio_path)

    return signal_np, sampling_rate

### Función `read_audio_by_id`

La función `read_audio_by_id()` permite recuperar la señal de audio a partir de un `audio_id`.

Para evitar búsquedas lentas dentro del dataset, se utiliza un diccionario auxiliar:

$$
audio\_id \longrightarrow índice
$$

Este diccionario permite localizar rápidamente el registro correspondiente dentro del dataset de Hugging Face.

Si el `audio_id` no existe en el diccionario, la función detiene el proceso y lanza un error controlado.

In [31]:

# Función para obtener señal de audio a partir de audio_id


def read_audio_by_id(audio_id, dataset, audio_id_to_index):
    """
    Lee la señal de audio correspondiente a un audio_id.

    Parámetros
    ----------
    audio_id : str
        Identificador único del audio.

    dataset : datasets.Dataset
        Dataset de Hugging Face con la columna audio en decode=False.

    audio_id_to_index : dict
        Diccionario que mapea audio_id a índice dentro del dataset.

    Retorna
    -------
    signal_np : np.ndarray
        Señal de audio en formato NumPy.

    sampling_rate : int
        Frecuencia de muestreo del audio.
    """

    # Verificamos que el audio_id exista dentro del índice.
    if audio_id not in audio_id_to_index:
        raise ValueError(f"No se encontró audio_id en el dataset: {audio_id}")

    # Recuperamos el índice asociado al audio_id.
    item_index = audio_id_to_index[audio_id]

    # Extraemos el registro individual del dataset.
    item = dataset[item_index]

    # Leemos la señal de audio desde el registro.
    signal_np, sampling_rate = read_audio_from_hf_item(item)

    return signal_np, sampling_rate

In [26]:

# Prueba de lectura de audio desde Hugging Face usando audio_id


sample_audio_id = df_enrollment.iloc[0]["audio_id"]

signal_np, sampling_rate = read_audio_by_id(
    audio_id=sample_audio_id,
    dataset=ciempiess,
    audio_id_to_index=audio_id_to_index
)

print("Audio ID:", sample_audio_id)
print("Tipo de señal:", type(signal_np))
print("Forma de señal:", signal_np.shape)
print("Frecuencia de muestreo:", sampling_rate)
print("Duración:", len(signal_np) / sampling_rate, "segundos")

Audio ID: CMPL_F_01_12MAB_00651
Tipo de señal: <class 'numpy.ndarray'>
Forma de señal: (90036,)
Frecuencia de muestreo: 16000
Duración: 5.62725 segundos


In [27]:

# Conversión de señal NumPy a tensor de PyTorch


# Convertimos la señal a tensor float32.
signal_tensor = torch.tensor(signal_np, dtype=torch.float32)

# Si la señal viene como [T], agregamos dimensión de batch: [1, T].
if signal_tensor.ndim == 1:
    signal_tensor = signal_tensor.unsqueeze(0)

# Si la señal viniera estéreo como [T, C], la convertimos a mono.
elif signal_tensor.ndim == 2:
    signal_tensor = signal_tensor.mean(dim=1).unsqueeze(0)

print("Forma del tensor:", signal_tensor.shape)
print("Tipo:", signal_tensor.dtype)

Forma del tensor: torch.Size([1, 90036])
Tipo: torch.float32


In [28]:

# Extracción de embedding x-vector desde audio cargado por Hugging Face


with torch.no_grad():
    sample_embedding = classifier.encode_batch(
        signal_tensor.to(device)
    )

sample_embedding_np = sample_embedding.squeeze().detach().cpu().numpy()

print("Forma del embedding:", sample_embedding_np.shape)
print("Primeros 10 valores:")
print(sample_embedding_np[:10])

Forma del embedding: (512,)
Primeros 10 valores:
[-21.346788     0.81090015   8.896874    13.991695     7.0598316
 -10.242589   -19.330435   -11.283061    12.876003     8.10391   ]


In [29]:

# Validación del embedding generado


assert isinstance(sample_embedding_np, np.ndarray), "El embedding no está en formato NumPy."
assert sample_embedding_np.ndim == 1, "El embedding no es un vector unidimensional."
assert not np.isnan(sample_embedding_np).any(), "El embedding contiene NaN."
assert not np.isinf(sample_embedding_np).any(), "El embedding contiene infinitos."

print("Embedding generado correctamente desde CIEMPIESS vía Hugging Face.")
print("Dimensión del embedding:", sample_embedding_np.shape[0])

Embedding generado correctamente desde CIEMPIESS vía Hugging Face.
Dimensión del embedding: 512


## 04. Funciones reutilizables para extracción de embeddings x-vector

En esta etapa se construirán funciones reutilizables para extraer embeddings x-vector a partir de los audios de CIEMPIESS Light.

El objetivo es construir una función que reciba un `audio_id`, busque el audio correspondiente en el dataset de Hugging Face, prepare la señal y devuelva el embedding.

### Función `prepare_signal_for_speechbrain`

La función convierte una señal de audio en formato NumPy a un tensor de PyTorch compatible con SpeechBrain.

Una señal mono suele representarse como:

$$
x \in \mathbb{R}^{T}
$$

donde $T$ es el número de muestras.

Sin embargo, el modelo espera una entrada con dimensión de batch:

$$
X \in \mathbb{R}^{B \times T}
$$

donde:

- $B$ es el tamaño del batch;
- $T$ es el número de muestras.

Para un solo audio:

$$
B = 1
$$

por lo que la forma esperada es:

$$
[1, T]
$$

La operación `unsqueeze(0)` agrega esta dimensión adicional.

In [32]:


def prepare_signal_for_speechbrain(signal_np, device="cpu"):
    """
    Convierte una señal de audio en formato NumPy a un tensor de PyTorch
    compatible con el modelo de SpeechBrain.

    Parámetros
    ----------
    signal_np : np.ndarray
        Señal de audio leída desde el dataset.

    device : str
        Dispositivo donde se colocará el tensor.
        Puede ser "cpu" o "cuda".

    Retorna
    -------
    signal_tensor : torch.Tensor
        Tensor de PyTorch con forma [1, T], listo para ser procesado
        por SpeechBrain.
    """

    # Convertimos el arreglo de NumPy a tensor de PyTorch en precisión float32.
    signal_tensor = torch.tensor(signal_np, dtype=torch.float32)

    # Caso 1: audio mono con forma [T].
    # Se agrega dimensión de batch para obtener [1, T].
    if signal_tensor.ndim == 1:
        signal_tensor = signal_tensor.unsqueeze(0)

    # Caso 2: audio multicanal con forma [T, C].
    # Se promedian los canales para obtener una señal mono.
    elif signal_tensor.ndim == 2:
        signal_tensor = signal_tensor.mean(dim=1).unsqueeze(0)

    # Si la señal tiene más dimensiones, no corresponde al caso esperado.
    else:
        raise ValueError(f"Forma de señal no esperada: {signal_tensor.shape}")

    # Movemos el tensor al dispositivo seleccionado: CPU o GPU.
    signal_tensor = signal_tensor.to(device)

    return signal_tensor

### Función `extract_xvector_embedding_from_audio_id`

Esta es la función principal de la etapa.

Su objetivo es transformar un `audio_id` en un embedding x-vector.

El proceso interno es:

1. Buscar el audio dentro del dataset de Hugging Face.
2. Leer la señal acústica.
3. Convertir la señal a tensor de PyTorch.
4. Enviar el tensor al modelo x-vector.
5. Convertir el resultado a un arreglo de NumPy.

Se utiliza `torch.no_grad()` porque el modelo se usa únicamente en modo inferencia.

In [ ]:
# Extraer embedding x-vector desde audio_id


def extract_xvector_embedding_from_audio_id(
    audio_id,
    dataset,
    audio_id_to_index,
    classifier,
    device="cpu",
    expected_sampling_rate=16000
):
    """
    Extrae el embedding x-vector correspondiente a un audio_id.

    Parámetros
    ----------
    audio_id : str
        Identificador único del audio.

    dataset : datasets.Dataset
        Dataset de Hugging Face que contiene los audios.

    audio_id_to_index : dict
        Diccionario que mapea audio_id a índice dentro del dataset.

    classifier : speechbrain.inference.speaker.EncoderClassifier
        Modelo preentrenado de SpeechBrain usado para extraer embeddings.

    device : str, default="cpu"
        Dispositivo de ejecución. Puede ser "cpu" o "cuda".

    expected_sampling_rate : int, default=16000
        Frecuencia de muestreo esperada. Para CIEMPIESS Light se espera 16000 Hz.

    Retorna
    -------
    embedding_np : np.ndarray
        Embedding x-vector como arreglo NumPy unidimensional.
    """

    # 1. Leer señal de audio desde Hugging Face usando audio_id

    signal_np, sampling_rate = read_audio_by_id(
        audio_id=audio_id,
        dataset=dataset,
        audio_id_to_index=audio_id_to_index
    )


    # 2. Validar frecuencia de muestreo

    if sampling_rate != expected_sampling_rate:
        raise ValueError(
            f"Frecuencia de muestreo inesperada para {audio_id}: "
            f"{sampling_rate}. Se esperaba {expected_sampling_rate}."
        )


    # 3. Preparar señal como tensor compatible con SpeechBrain

    signal_tensor = prepare_signal_for_speechbrain(
        signal_np=signal_np,
        device=device
    )

    # 4. Extraer embedding usando el modelo x-vector

    # torch.no_grad() desactiva el cálculo de gradientes porque
    # estamos haciendo inferencia, no entrenamiento.
    with torch.no_grad():
        embedding = classifier.encode_batch(signal_tensor)


    # 5. Convertir embedding a NumPy

    # squeeze(): elimina dimensiones unitarias.
    # detach(): separa el tensor del grafo computacional.
    # cpu(): mueve el tensor a CPU si estaba en GPU.
    # numpy(): convierte el tensor a arreglo NumPy.
    embedding_np = embedding.squeeze().detach().cpu().numpy()


    # 6. Validaciones básicas del embedding

    if embedding_np.ndim != 1:
        raise ValueError(
            f"El embedding de {audio_id} no es un vector 1D. "
            f"Forma obtenida: {embedding_np.shape}"
        )

    if np.isnan(embedding_np).any():
        raise ValueError(f"El embedding de {audio_id} contiene valores NaN.")

    if np.isinf(embedding_np).any():
        raise ValueError(f"El embedding de {audio_id} contiene valores infinitos.")

    return embedding_np

In [34]:

# Prueba formal de la función principal con df_enrollment


sample_audio_id = df_enrollment.iloc[0]["audio_id"]

sample_embedding = extract_xvector_embedding_from_audio_id(
    audio_id=sample_audio_id,
    dataset=ciempiess,
    audio_id_to_index=audio_id_to_index,
    classifier=classifier,
    device=device,
    expected_sampling_rate=16000
)

print("Audio ID:", sample_audio_id)
print("Tipo del embedding:", type(sample_embedding))
print("Forma del embedding:", sample_embedding.shape)
print("Primeros 10 valores:")
print(sample_embedding[:10])

Audio ID: CMPL_F_01_12MAB_00651
Tipo del embedding: <class 'numpy.ndarray'>
Forma del embedding: (512,)
Primeros 10 valores:
[-21.346634     0.81081384   8.896763    13.991639     7.05981
 -10.242543   -19.330276   -11.283158    12.87593      8.103908  ]


In [36]:

# Prueba de extracción con varios audios de enrollment


sample_audio_ids = df_enrollment["audio_id"].head(5).tolist()

sample_embeddings = {}

for audio_id in tqdm(sample_audio_ids, desc="Extrayendo embeddings de prueba"):
    embedding = extract_xvector_embedding_from_audio_id(
        audio_id=audio_id,
        dataset=ciempiess,
        audio_id_to_index=audio_id_to_index,
        classifier=classifier,
        device=device,
        expected_sampling_rate=16000
    )

    sample_embeddings[audio_id] = embedding

print("Número de embeddings extraídos:", len(sample_embeddings))

for audio_id, emb in sample_embeddings.items():
    print(audio_id, emb.shape)

Extrayendo embeddings de prueba: 100%|██████████| 5/5 [00:00<00:00, 12.44it/s]

Número de embeddings extraídos: 5
CMPL_F_01_12MAB_00651 (512,)
CMPL_F_02_02MAB_00241 (512,)
CMPL_F_03_09MAB_00142 (512,)
CMPL_F_04_12ANG_00215 (512,)
CMPL_F_05_01CAR_00021 (512,)


## 05. Extracción de embeddings x-vector para enrollment

En esta etapa se extraerá un embedding x-vector para cada audio del conjunto de enrollment.

Dado que el protocolo experimental utiliza un audio de enrollment por cada locutor, el embedding extraído de ese audio será usado como representación de referencia del locutor.

El resultado será un nuevo DataFrame llamado `df_enrollment_embeddings`, que contendrá los metadatos originales junto con el embedding correspondiente.

In [ ]:
# Crear copia de trabajo para embeddings de enrollment


df_enrollment_embeddings = df_enrollment.copy()


In [43]:
# Extracción de embeddings x-vector para audios de enrollment


# Lista donde se almacenará el embedding de cada audio.
enrollment_embeddings = []

# Lista auxiliar para registrar errores si algún audio falla.
enrollment_errors = []

# Recorremos cada fila del DataFrame de enrollment.
for row in tqdm(
    df_enrollment_embeddings.itertuples(index=False),
    total=len(df_enrollment_embeddings),
    desc="Extrayendo embeddings de enrollment"
):
    # Extraemos el audio_id de la fila actual.
    audio_id = row.audio_id

    try:
        # Extraemos el embedding x-vector usando la función principal.
        embedding = extract_xvector_embedding_from_audio_id(
            audio_id=audio_id,
            dataset=ciempiess,
            audio_id_to_index=audio_id_to_index,
            classifier=classifier,
            device=device,
            expected_sampling_rate=16000
        )

        # Guardamos el embedding en la lista principal.
        enrollment_embeddings.append(embedding)

    except Exception as e:
        # Si ocurre un error, guardamos None como embedding
        # y registramos información del error.
        enrollment_embeddings.append(None)

        enrollment_errors.append(
            {
                "audio_id": audio_id,
                "error": str(e)
            }
        )

# Añadimos la lista de embeddings como nueva columna.
df_enrollment_embeddings["xvector_embedding"] = enrollment_embeddings

print("Extracción de embeddings de enrollment finalizada.")
print("Número de errores:", len(enrollment_errors))

Extrayendo embeddings de enrollment: 100%|██████████| 87/87 [00:07<00:00, 11.89it/s]

Extracción de embeddings de enrollment finalizada.
Número de errores: 0


### Matriz de embeddings de enrollment

Aunque los embeddings se almacenan como una columna dentro del DataFrame, para cálculos posteriores conviene construir una matriz numérica.

Si existen $N$ audios de enrollment y cada embedding tiene dimensión $d$, entonces la matriz de embeddings es:

$$
Z^{enroll} =
\begin{bmatrix}
 & z_1^{enroll} &  \\
 & z_2^{enroll} &  \\
  & \vdots &   \\
 & z_N^{enroll} & 
\end{bmatrix}
\in \mathbb{R}^{N \times d}
$$

Cada fila representa el embedding de un audio de enrollment.

In [ ]:
enrollment_embedding_matrix = np.vstack(
    df_enrollment_embeddings["xvector_embedding"].values
)

print("Forma de la matriz de embeddings de enrollment:")
print(enrollment_embedding_matrix.shape)

Forma de la matriz de embeddings de enrollment:
(87, 512)


In [46]:
# Metadatos asociados a la matriz de enrollment


df_enrollment_embedding_metadata = df_enrollment_embeddings[
    [
        "audio_id",
        "speaker_id",
        "gender",
        "audio_path",
        "sampling_rate",
        "num_samples",
        "duration_sec",
        "duration_range",
        "split"
    ]
].copy()

print("Dimensión de metadatos de enrollment:")
print(df_enrollment_embedding_metadata.shape)

display(df_enrollment_embedding_metadata.head())

Dimensión de metadatos de enrollment:
(87, 9)


,audio_id,speaker_id,gender,audio_path,sampling_rate,num_samples,duration_sec,duration_range,split
0,CMPL_F_01_12MAB_00651,F_01,female,CMPL_F_01_12MAB_00651.flac,16000,90036,5.627250,5 a 10 s,enrollment
1,CMPL_F_02_02MAB_00241,F_02,female,CMPL_F_02_02MAB_00241.flac,16000,56244,3.515250,2 a 5 s,enrollment
2,CMPL_F_03_09MAB_00142,F_03,female,CMPL_F_03_09MAB_00142.flac,16000,120857,7.553562,5 a 10 s,enrollment
3,CMPL_F_04_12ANG_00215,F_04,female,CMPL_F_04_12ANG_00215.flac,16000,33306,2.081625,2 a 5 s,enrollment
4,CMPL_F_05_01CAR_00021,F_05,female,CMPL_F_05_01CAR_00021.flac,16000,89708,5.606750,5 a 10 s,enrollment


In [ ]:
# Como en este protocolo hay un solo audio de enrollment por locutor, podemos construir directamente un diccionario:
speaker_enrollment_embedding_dict = dict(
    zip(
        df_enrollment_embeddings["speaker_id"],
        df_enrollment_embeddings["xvector_embedding"]
    )
)

print("Número de locutores en el diccionario:", len(speaker_enrollment_embedding_dict))

Número de locutores en el diccionario: 87


## 06. Guardado de embeddings de enrollment

Los embeddings se guardarán en tres formatos:

1. `.pkl`: conserva el DataFrame completo, incluyendo la columna con arreglos NumPy.
2. `.npy`: guarda únicamente la matriz numérica de embeddings.
3. `.csv`: guarda los metadatos asociados a cada fila de la matriz.

Esto permite reutilizar los embeddings sin tener que volver a procesar los audios.

In [53]:

# Guardado de embeddings de enrollment


# Crear carpeta de salida.
output_dir = Path("outputs")
embedding_output_dir = output_dir / "embeddings"
embedding_output_dir.mkdir(parents=True, exist_ok=True)

# Rutas de salida.
enrollment_pkl_path = embedding_output_dir / "xvector_enrollment_embeddings.pkl"
enrollment_matrix_path = embedding_output_dir / "xvector_enrollment_matrix.npy"
enrollment_metadata_path = embedding_output_dir / "xvector_enrollment_metadata.csv"

# Guardar DataFrame completo con embeddings.
df_enrollment_embeddings.to_pickle(enrollment_pkl_path)

# Guardar matriz numérica.
np.save(enrollment_matrix_path, enrollment_embedding_matrix)

# Guardar metadatos asociados.
df_enrollment_embedding_metadata.to_csv(enrollment_metadata_path, index=False)

print("Archivos guardados:")
print(enrollment_pkl_path)
print(enrollment_matrix_path)
print(enrollment_metadata_path)

Archivos guardados:
outputs\embeddings\xvector_enrollment_embeddings.pkl
outputs\embeddings\xvector_enrollment_matrix.npy
outputs\embeddings\xvector_enrollment_metadata.csv


## 07. Extracción de embeddings x-vector para test

En esta etapa se extraerán embeddings x-vector para todos los audios del conjunto de test.

A diferencia del conjunto de enrollment, que contiene 87 audios, el conjunto de test contiene 13,165 audios, por lo que esta etapa es más costosa computacionalmente.

El resultado principal será un DataFrame llamado `df_test_embeddings`, que contendrá los metadatos originales junto con el embedding x-vector correspondiente.

In [56]:

# Crear copia de trabajo para embeddings de test
df_test_embeddings = df_test.copy()


In [59]:
# Rutas de guardado para embeddings de test

# Como esta extracción puede tardar, guardaremos un archivo temporal con los avances.

output_dir = Path("outputs")
embedding_output_dir = output_dir / "embeddings"
embedding_output_dir.mkdir(parents=True, exist_ok=True)

test_checkpoint_path = embedding_output_dir / "xvector_test_embeddings_checkpoint.pkl"
test_errors_path = embedding_output_dir / "xvector_test_embedding_errors.csv"

print("Checkpoint de test:")
print(test_checkpoint_path)

print("\nArchivo de errores:")
print(test_errors_path)

Checkpoint de test:
outputs\embeddings\xvector_test_embeddings_checkpoint.pkl

Archivo de errores:
outputs\embeddings\xvector_test_embedding_errors.csv


### Función auxiliar para procesar un DataFrame completo

La función `extract_embeddings_for_dataframe()` aplica la función de extracción de embeddings a todos los audios contenidos en un DataFrame.

El flujo es:

$$
audio\_id_i \longrightarrow x_i \longrightarrow z_i
$$

para cada fila $i$ del DataFrame.

Como esta operación puede tardar, se incluyen tres elementos importantes:

1. **Barra de progreso:** permite monitorear el avance.
2. **Manejo de errores:** evita que un solo audio detenga todo el proceso.
3. **Guardado intermedio:** permite recuperar resultados parciales si la ejecución se interrumpe.

El resultado es una copia del DataFrame original con una nueva columna llamada `xvector_embedding`.

In [57]:

# Función auxiliar: extraer embeddings para un DataFrame


def extract_embeddings_for_dataframe(
    df,
    dataset,
    audio_id_to_index,
    classifier,
    device="cpu",
    expected_sampling_rate=16000,
    embedding_column="xvector_embedding",
    checkpoint_path=None,
    save_every=500,
    desc="Extrayendo embeddings"
):
    """
    Extrae embeddings x-vector para todos los audios de un DataFrame.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame que debe contener una columna llamada 'audio_id'.

    dataset : datasets.Dataset
        Dataset de Hugging Face que contiene los audios.

    audio_id_to_index : dict
        Diccionario que mapea audio_id a índice dentro del dataset.

    classifier : speechbrain.inference.speaker.EncoderClassifier
        Modelo x-vector preentrenado de SpeechBrain.

    device : str, default="cpu"
        Dispositivo de ejecución: "cpu" o "cuda".

    expected_sampling_rate : int, default=16000
        Frecuencia de muestreo esperada.

    embedding_column : str, default="xvector_embedding"
        Nombre de la columna donde se guardarán los embeddings.

    checkpoint_path : str or pathlib.Path, default=None
        Ruta donde se guardará periódicamente el DataFrame parcial.

    save_every : int, default=500
        Frecuencia de guardado intermedio. Cada `save_every` audios
        se guarda un checkpoint.

    desc : str, default="Extrayendo embeddings"
        Texto descriptivo para la barra de progreso.

    Retorna
    -------
    df_out : pandas.DataFrame
        Copia del DataFrame original con una columna adicional de embeddings.

    errors : list of dict
        Lista con los errores encontrados durante el procesamiento.
    """

    # Creamos una copia para no modificar el DataFrame original.
    df_out = df.copy()

    # Lista donde se guardarán los embeddings.
    embeddings = []

    # Lista donde se registrarán errores.
    errors = []

    # Iteramos sobre las filas del DataFrame.
    for idx, row in tqdm(
        enumerate(df_out.itertuples(index=False)),
        total=len(df_out),
        desc=desc
    ):
        # Extraemos el audio_id de la fila actual.
        audio_id = row.audio_id

        try:
            # Extraemos embedding con la función principal.
            embedding = extract_xvector_embedding_from_audio_id(
                audio_id=audio_id,
                dataset=dataset,
                audio_id_to_index=audio_id_to_index,
                classifier=classifier,
                device=device,
                expected_sampling_rate=expected_sampling_rate
            )

            embeddings.append(embedding)

        except Exception as e:
            # Si hay error, guardamos None y registramos el problema.
            embeddings.append(None)

            errors.append(
                {
                    "row_index": idx,
                    "audio_id": audio_id,
                    "error": str(e)
                }
            )

        # Guardado intermedio.
        if checkpoint_path is not None and (idx + 1) % save_every == 0:
            df_partial = df_out.iloc[: idx + 1].copy()
            df_partial[embedding_column] = embeddings

            df_partial.to_pickle(checkpoint_path)

            print(f"Checkpoint guardado en audio {idx + 1}: {checkpoint_path}")

    # Al terminar, añadimos todos los embeddings al DataFrame de salida.
    df_out[embedding_column] = embeddings

    # Guardado final del checkpoint, si se indicó ruta.
    if checkpoint_path is not None:
        df_out.to_pickle(checkpoint_path)
        print(f"Checkpoint final guardado en: {checkpoint_path}")

    return df_out, errors

In [60]:
# Extracción de embeddings x-vector para audios de test


df_test_embeddings, test_errors = extract_embeddings_for_dataframe(
    df=df_test_embeddings,
    dataset=ciempiess,
    audio_id_to_index=audio_id_to_index,
    classifier=classifier,
    device=device,
    expected_sampling_rate=16000,
    embedding_column="xvector_embedding",
    checkpoint_path=test_checkpoint_path,
    save_every=500,
    desc="Extrayendo embeddings de test"
)

print("Extracción de embeddings de test finalizada.")
print("Número de errores:", len(test_errors))

Extrayendo embeddings de test:   4%|▍         | 501/13165 [00:42<17:27, 12.09it/s]

Checkpoint guardado en audio 500: outputs\embeddings\xvector_test_embeddings_checkpoint.pkl


Extrayendo embeddings de test:   8%|▊         | 1002/13165 [01:30<18:34, 10.92it/s]

Checkpoint guardado en audio 1000: outputs\embeddings\xvector_test_embeddings_checkpoint.pkl


Extrayendo embeddings de test:  11%|█▏        | 1500/13165 [02:22<20:18,  9.57it/s]

...

Checkpoint guardado en audio 12500: outputs\embeddings\xvector_test_embeddings_checkpoint.pkl


Extrayendo embeddings de test:  99%|█████████▉| 13001/13165 [20:28<00:23,  6.89it/s]

Checkpoint guardado en audio 13000: outputs\embeddings\xvector_test_embeddings_checkpoint.pkl


Extrayendo embeddings de test: 100%|██████████| 13165/13165 [20:40<00:00, 10.61it/s]


Checkpoint final guardado en: outputs\embeddings\xvector_test_embeddings_checkpoint.pkl
Extracción de embeddings de test finalizada.
Número de errores: 0


In [61]:
# Construcción de matriz de embeddings de test


test_embedding_matrix = np.vstack(
    df_test_embeddings["xvector_embedding"].values
)

print("Forma de la matriz de embeddings de test:")
print(test_embedding_matrix.shape)

Forma de la matriz de embeddings de test:
(13165, 512)


In [63]:
# Diccionario test_audio_id -> embedding de test

test_audio_embedding_dict = dict(
    zip(
        df_test_embeddings["audio_id"],
        df_test_embeddings["xvector_embedding"]
    )
)

print("Número de audios en el diccionario de test:", len(test_audio_embedding_dict))

Número de audios en el diccionario de test: 13165


In [67]:
# Metadatos asociados a la matriz de embeddings de test


df_test_embedding_metadata = df_test_embeddings[
    [
        "audio_id",
        "speaker_id",
        "gender",
        "audio_path",
        "sampling_rate",
        "num_samples",
        "duration_sec",
        "duration_range",
        "split"
    ]
].copy()

print("Dimensión de metadatos de test:")
print(df_test_embedding_metadata.shape)

Dimensión de metadatos de test:
(13165, 9)


In [68]:
# ============================================================
# Guardado de embeddings de test
# ============================================================

test_pkl_path = embedding_output_dir / "xvector_test_embeddings.pkl"
test_matrix_path = embedding_output_dir / "xvector_test_matrix.npy"
test_metadata_path = embedding_output_dir / "xvector_test_metadata.csv"

# Guardar DataFrame completo con embeddings.
df_test_embeddings.to_pickle(test_pkl_path)

# Guardar matriz numérica.
np.save(test_matrix_path, test_embedding_matrix)

# Guardar metadatos asociados.
df_test_embedding_metadata.to_csv(test_metadata_path, index=False)

print("Archivos guardados:")
print(test_pkl_path)
print(test_matrix_path)
print(test_metadata_path)

Archivos guardados:
outputs\embeddings\xvector_test_embeddings.pkl
outputs\embeddings\xvector_test_matrix.npy
outputs\embeddings\xvector_test_metadata.csv


## Documentación de formatos de almacenamiento de embeddings

Durante la extracción de embeddings x-vector se generaron y guardaron distintos archivos para almacenar tanto los vectores numéricos como sus metadatos asociados.

Se utilizaron tres formatos principales:

1. `.pkl`
2. `.npy`
3. `.csv`

Cada formato cumple una función distinta dentro del flujo experimental.

El objetivo de guardar estos archivos es evitar recalcular los embeddings en etapas posteriores, ya que la extracción con modelos neuronales puede ser costosa en tiempo y recursos computacionales.

### Formato `.pkl`

El formato `.pkl`, abreviatura de *pickle*, permite guardar objetos completos de Python.

En este proyecto se utiliza para guardar DataFrames de pandas que contienen tanto metadatos como embeddings.

Por ejemplo:

```python
df_enrollment_embeddings.to_pickle("xvector_enrollment_embeddings.pkl")
df_test_embeddings.to_pickle("xvector_test_embeddings.pkl")

### Formato `.npy`

El formato `.npy` es un formato binario propio de NumPy diseñado para almacenar arreglos numéricos.

En este proyecto se utiliza para guardar matrices de embeddings.

Por ejemplo:

```python
np.save("xvector_enrollment_matrix.npy", enrollment_embedding_matrix)
np.save("xvector_test_matrix.npy", test_embedding_matrix)

En este proyecto, la estrategia será:

1. Usar `.pkl` para recuperar rápidamente los DataFrames completos.
2. Usar `.npy` para trabajar con matrices de embeddings en scoring.
3. Usar `.csv` para mantener trazabilidad de cada embedding.

En la siguiente fase, correspondiente al cálculo de scores, será necesario recuperar embeddings a partir de los identificadores contenidos en `df_trials`.

## 08. Preparación de trials para scoring

En esta etapa se preparará la tabla `df_trials` para la fase posterior de scoring.

Cada fila de `df_trials` representa una comparación entre un locutor de enrollment y un audio de test.

El objetivo de esta etapa es verificar que cada trial pueda recuperar correctamente ambos embeddings.

### Diccionarios de embeddings

Para conectar los trials con los embeddings, se construyen dos diccionarios:

$$
speaker\_id \longrightarrow z_s^{enroll}
$$

$$
test\_audio\_id \longrightarrow z_j^{test}
$$

El primer diccionario permite recuperar el embedding de referencia del locutor de enrollment.

El segundo diccionario permite recuperar el embedding correspondiente al audio de test.

Estos diccionarios son útiles porque permiten hacer búsquedas rápidas a partir de identificadores.

In [69]:
# Construcción de diccionarios de embeddings

# Diccionario: speaker_id -> embedding de enrollment
speaker_enrollment_embedding_dict = dict(
    zip(
        df_enrollment_embeddings["speaker_id"],
        df_enrollment_embeddings["xvector_embedding"]
    )
)

# Diccionario: test_audio_id -> embedding de test
test_audio_embedding_dict = dict(
    zip(
        df_test_embeddings["audio_id"],
        df_test_embeddings["xvector_embedding"]
    )
)

print("Número de embeddings de enrollment por speaker:", len(speaker_enrollment_embedding_dict))
print("Número de embeddings de test por audio:", len(test_audio_embedding_dict))

Número de embeddings de enrollment por speaker: 87
Número de embeddings de test por audio: 13165


In [76]:
# Crear copia de trabajo para trials con embeddings

df_trials_ready = df_trials.copy()

print("Dimensión de df_trials_ready:", df_trials_ready.shape)


Dimensión de df_trials_ready: (26330, 13)


### Agregar embeddings a la tabla de trials

Una vez validados los diccionarios, se agregan dos columnas a `df_trials_ready`:

- `enrollment_embedding`;
- `test_embedding`.

Estas columnas contienen los vectores que serán comparados en la fase de scoring.

Para cada trial:

$$
(z_i^{enroll}, z_j^{test})
$$


In [83]:
# Agregar embeddings a df_trials_ready


df_trials_ready["enrollment_embedding"] = df_trials_ready["enrollment_speaker_id"].map(
    speaker_enrollment_embedding_dict
)

df_trials_ready["test_embedding"] = df_trials_ready["test_audio_id"].map(
    test_audio_embedding_dict
)



### Versión ligera de trials

La tabla `df_trials_ready` contiene embeddings completos en cada fila, lo cual puede ocupar bastante memoria.

Por esta razón se construye también una versión ligera llamada `df_trials_scoring_metadata`.

Esta versión no contiene los vectores, pero conserva los identificadores necesarios para recuperar los embeddings desde diccionarios o matrices.

La versión ligera es útil para guardar trazabilidad experimental sin duplicar información pesada.

In [86]:

# Crear versión ligera de trials para scoring


df_trials_scoring_metadata = df_trials_ready[
    [
        "trial_id",
        "enrollment_speaker_id",
        "enrollment_audio_id",
        "test_speaker_id",
        "test_audio_id",
        "gender_enrollment",
        "gender_test",
        "trial_type",
        "same_speaker",
        "same_gender",
        "label"
    ]
].copy()

print("Dimensión de df_trials_scoring_metadata:")
print(df_trials_scoring_metadata.shape)


Dimensión de df_trials_scoring_metadata:
(26330, 11)


In [87]:
# Guardado de trials preparados para scoring


trials_ready_pkl_path = embedding_output_dir / "xvector_trials_ready.pkl"
trials_scoring_metadata_path = embedding_output_dir / "xvector_trials_scoring_metadata.csv"

df_trials_ready.to_pickle(trials_ready_pkl_path)

df_trials_scoring_metadata.to_csv(
    trials_scoring_metadata_path,
    index=False
)

print("Archivos guardados:")
print(trials_ready_pkl_path)
print(trials_scoring_metadata_path)

Archivos guardados:
outputs\embeddings\xvector_trials_ready.pkl
outputs\embeddings\xvector_trials_scoring_metadata.csv
